# Day 5 — Solver Tuning Support + Literature Positioning

**Project:** Dark Store Placement + Integrated Forward & Reverse Logistics  
**Author:** Sneha  

**Depends on:**
- `data/master_df_v2.parquet` — with `dark_store_id` 
- `data/dark_stores_final.csv` — finalised dark store locations 

> **Note:**  `05_reverse_vrp.ipynb` runs solver tuning on his zones. This notebook runs the **same 3 strategies on 2 additional zones** (Sneha's support task) and documents the solver strategy decision.

**Outputs:**
- `outputs/solver_tuning_results.csv` — tuning comparison across strategies and zones
- Printed solver strategy decision (to document in `src/reverse_vrp.py`)
- `literature_positioning.docx` is generated separately (see report/ folder)

---
## 0. Setup

In [ ]:
import sys, os, time
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ortools.constraint_solver import routing_enums_pb2, pywrapcp

from src.route_parser import (
    build_distance_matrix,
    build_vrp_nodes,
    parse_solution,
    compute_routing_cost,
    VEHICLE_CAPACITY_G,
    VEHICLE_SPEED_KMH,
    FIXED_COST_PER_ROUTE,
    VAR_COST_PER_KM,
    SERVICE_TIME_MIN,
    SOLVER_TIME_LIMIT_S,
    MAX_CUSTOMERS_PER_ZONE,
    NUM_VEHICLES,
)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

DATA_DIR   = '../data'
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports OK.')
print(f'Solver time limit per zone : {SOLVER_TIME_LIMIT_S}s')
print(f'Max customers per zone     : {MAX_CUSTOMERS_PER_ZONE}')
print(f'Vehicle capacity           : {VEHICLE_CAPACITY_G/1000:.0f} kg')

---
## 1. Load data and build forward VRP zones

We use `master_df_v2.parquet` (available from Day 3).  
Note: Vybhav uses `master_df_v3.parquet` (which has `return_prob`) for reverse VRP tuning.  
For forward VRP tuning support, `master_df_v2` is sufficient.

In [ ]:
master_df   = pd.read_parquet(f'{DATA_DIR}/master_df_v2.parquet')
dark_stores = pd.read_csv(f'{DATA_DIR}/dark_stores_final.csv')

print(f'master_df shape  : {master_df.shape}')
print(f'Dark stores (K)  : {len(dark_stores)}')
print(f'dark_store_id values: {sorted(master_df["dark_store_id"].unique())}')

In [ ]:
# Build forward VRP zones (same as Pritam's forward VRP pipeline)
all_zones = build_vrp_nodes(
    master_df, dark_stores,
    max_per_zone=MAX_CUSTOMERS_PER_ZONE,
    seed=42
)

print(f'\nTotal zones built: {len(all_zones)}')

---
## 2. Select 2 zones for tuning

The task says: run the same 3 strategies on **2 more zones** to support Vybhav.  
We pick the 2 zones with the most customers (highest demand) — these are the hardest to solve and give the most meaningful tuning signal.

In [ ]:
# Pick 2 largest zones (most customers) for tuning
zones_sorted = sorted(all_zones, key=lambda z: z['n_customers'], reverse=True)
TUNE_ZONES   = zones_sorted[:2]

print('Zones selected for solver tuning:')
for z in TUNE_ZONES:
    print(f"  Zone {z['zone_id']}: {z['n_customers']} customers | "
          f"total demand = {z['demands'].sum()/1000:.1f} kg")

---
## 3. Run 3 solver strategies on both zones

**The 3 strategies** (same as in `05_reverse_vrp.ipynb`):
1. `PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH` — fast greedy start, GLS improvement
2. `PATH_CHEAPEST_ARC + SIMULATED_ANNEALING` — fast start, SA improvement (accepts worse moves)
3. `AUTOMATIC + GUIDED_LOCAL_SEARCH` — OR-Tools chooses start, GLS improvement

We measure: total distance (km), routing cost (R$), and solve time (seconds).

In [ ]:
STRATEGIES = [
    (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC,
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH,
        'PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH',
    ),
    (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC,
        routing_enums_pb2.LocalSearchMetaheuristic.SIMULATED_ANNEALING,
        'PATH_CHEAPEST_ARC + SIMULATED_ANNEALING',
    ),
    (
        routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC,
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH,
        'AUTOMATIC + GUIDED_LOCAL_SEARCH',
    ),
]

tuning_rows = []

for zone in TUNE_ZONES:
    print(f"\n{'='*55}")
    print(f"Zone {zone['zone_id']} ({zone['n_customers']} customers)")
    print('='*55)

    n       = len(zone['node_coords'])
    demands = zone['demands'].tolist()
    tw      = zone['time_windows']
    dist_m  = build_distance_matrix(zone['node_coords'])
    time_m  = np.rint(dist_m / (VEHICLE_SPEED_KMH * 1000 / 60)).astype(int)

    for fss, lsm, label in STRATEGIES:
        manager = pywrapcp.RoutingIndexManager(n, NUM_VEHICLES, 0)
        routing = pywrapcp.RoutingModel(manager)

        _dist_m = dist_m  # capture for closure
        def dist_cb(i, j, _d=_dist_m):
            return int(_d[manager.IndexToNode(i)][manager.IndexToNode(j)])
        dist_cb_idx = routing.RegisterTransitCallback(dist_cb)
        routing.SetArcCostEvaluatorOfAllVehicles(dist_cb_idx)

        _demands = demands
        def demand_cb(i, _dem=_demands):
            return _dem[manager.IndexToNode(i)]
        dem_cb_idx = routing.RegisterUnaryTransitCallback(demand_cb)
        routing.AddDimensionWithVehicleCapacity(
            dem_cb_idx, 0,
            [VEHICLE_CAPACITY_G] * NUM_VEHICLES,
            True, 'Capacity'
        )

        _time_m = time_m
        def time_cb(i, j, _t=_time_m):
            return int(_t[manager.IndexToNode(i)][manager.IndexToNode(j)]) + SERVICE_TIME_MIN
        time_cb_idx = routing.RegisterTransitCallback(time_cb)
        routing.AddDimension(time_cb_idx, 60, 1440, False, 'Time')
        time_dim = routing.GetDimensionOrDie('Time')
        for idx_node, (tw_open, tw_close) in enumerate(tw):
            ri = manager.NodeToIndex(idx_node)
            time_dim.CumulVar(ri).SetRange(tw_open, tw_close)

        params = pywrapcp.DefaultRoutingSearchParameters()
        params.first_solution_strategy = fss
        params.local_search_metaheuristic = lsm
        params.time_limit.seconds = SOLVER_TIME_LIMIT_S

        t0 = time.time()
        assignment = routing.SolveWithParameters(params)
        elapsed = round(time.time() - t0, 2)

        if assignment:
            routes_df, summary = parse_solution(
                manager, routing, assignment,
                zone['node_coords'], zone['node_ids'], dist_m
            )
            total_dist_km  = summary['total_distance_km']
            n_veh          = summary['n_vehicles_used']
            cost           = compute_routing_cost(n_veh, total_dist_km)
            solved         = True
        else:
            total_dist_km = cost = n_veh = 0.0
            solved = False

        row = {
            'zone_id':         zone['zone_id'],
            'n_customers':     zone['n_customers'],
            'strategy':        label,
            'solved':          solved,
            'total_dist_km':   round(total_dist_km, 2),
            'n_vehicles':      n_veh,
            'routing_cost_R$': round(cost, 2),
            'solve_time_s':    elapsed,
        }
        tuning_rows.append(row)
        status = '✓' if solved else '✗'
        print(f'  {status} {label:<45} | dist={total_dist_km:7.1f}km | cost=R${cost:7.1f} | t={elapsed}s')

tuning_df = pd.DataFrame(tuning_rows)
print(f'\nTuning complete. {len(tuning_df)} runs total.')

---
## 4. Identify the winning strategy

In [ ]:
summary_df = (
    tuning_df[tuning_df['solved']]
    .groupby('strategy')[['total_dist_km', 'routing_cost_R$', 'solve_time_s']]
    .mean().round(2)
    .sort_values('routing_cost_R$')
)

print('Average across both tuned zones (lower cost = better):')
print('='*65)
print(summary_df.to_string())
print('='*65)

best_strategy = summary_df.index[0]
print(f'\n🏆 Winner: {best_strategy}')
print()
print('This decision should be documented in src/reverse_vrp.py:')
print('  STRATEGY_LABEL = "' + best_strategy + '"')

---
## 5. Save outputs

In [ ]:
# Save tuning results CSV
tuning_path = f'{OUTPUT_DIR}/solver_tuning_results.csv'
tuning_df.to_csv(tuning_path, index=False)
print(f'Saved → {tuning_path}')

# Bar chart: cost and distance by strategy and zone
ZONE_COLOURS = ['#457B9D', '#E63946']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['total_dist_km', 'routing_cost_R$', 'solve_time_s']
titles  = ['Total distance (km)', 'Routing cost (R$)', 'Solve time (s)']

for ax, metric, title in zip(axes, metrics, titles):
    solved_df = tuning_df[tuning_df['solved']]
    pivot = solved_df.pivot(index='strategy', columns='zone_id', values=metric)
    pivot.plot(kind='bar', ax=ax, color=ZONE_COLOURS[:len(pivot.columns)],
               edgecolor='white', width=0.65)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelrotation=18)
    ax.set_xticklabels([s.replace(' + ', '\n+  ') for s in pivot.index], fontsize=8, ha='right')
    ax.legend(title='Zone', fontsize=9)

# Highlight winner
for ax in axes:
    labels = [t.get_text().replace('\n+  ', ' + ') for t in ax.get_xticklabels()]
    for i, label in enumerate(labels):
        if label == best_strategy:
            ax.get_xticklabels()[i].set_color('green')
            ax.get_xticklabels()[i].set_fontweight('bold')

plt.suptitle(
    f'Solver Tuning — 3 Strategies × 2 Zones\nWinner: {best_strategy}',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
chart_path = f'{OUTPUT_DIR}/solver_tuning_chart.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {chart_path}')

---
## 6. Day 5 Checklist

In [ ]:
files_to_check = [
    (f'{OUTPUT_DIR}/solver_tuning_results.csv', 'solver_tuning_results.csv'),
    (f'{OUTPUT_DIR}/solver_tuning_chart.png',   'solver_tuning_chart.png'),
]

print('Day 5 Notebook Checklist')
print('='*50)
for path, label in files_to_check:
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {label}')

print()
print('Solver strategy decision:')
print(f'  🏆 Winner: {best_strategy}')
print()
print('literature_positioning.docx → generated separately (see report/ folder)')
print()
print('🎉 Day 5 solver tuning support complete!')